# 04 - Selección Automática de Features con Lasso y SelectFromModel

Este notebook documenta el proceso automático de selección de features usando Lasso regularizado (o L1 si el problema es clasificación binaria) sobre los datos *ya transformados* por el pipeline anterior.

----

## 1. Cargar datos transformados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# Para reproducibilidad
RANDOM_STATE = 42

# Ajustar rutas según output anterior del pipeline:
X_train = pd.read_csv('proyecto_pipeline_machine_learning_muestra/X_train_transformed.csv')
X_test = pd.read_csv('proyecto_pipeline_machine_learning_muestra/X_test_transformed.csv')

print('Shape X_train:', X_train.shape)
print('Shape X_test:', X_test.shape)

feature_names = X_train.columns.tolist()

## 2. Preparar el target (`SalePrice`)

In [ ]:
# Usamos SalePrice original para regresión, log1p si sigue sesgado.
y_train = pd.read_csv('proyecto_pipeline_machine_learning_muestra/y_train.csv').squeeze()

es_continuo = y_train.nunique() > 10
if es_continuo:
    skew = y_train.skew()
    print(f'Sesgo original SalePrice: {skew:.2f}')
    if skew > 0.75:
        print('Target con alto sesgo: se aplica log1p.')
        y_train = np.log1p(y_train)
        target_transform = 'log1p'
    else:
        target_transform = 'none'
else:
    print('Target parece binario/categórico, clasificación.')
    # No transformar
target_type = 'regression' if es_continuo else 'binary_classification'

## 3. Determinar tipo y configuración de modelo L1

In [ ]:
# Ajustar alpha/C según nº de features
def ajustar_alpha(n_features, tipo):
    if n_features > 300:
        return 0.025 if tipo == 'regression' else 0.03     # ajustar según dominio
    elif n_features > 100:
        return 0.01 if tipo == 'regression' else 0.02
    elif n_features > 40:
        return 0.005 if tipo == 'regression' else 0.01
    else:
        return 0.001 if tipo == 'regression' else 0.5      # C para clasificación

n_features = X_train.shape[1]
if target_type == 'regression':
    alpha = ajustar_alpha(n_features, 'regression')
    print(f'Modelo: Lasso (regresión). Alpha usado: {alpha}')
    model = Lasso(alpha=alpha, random_state=RANDOM_STATE, max_iter=10000)
else:
    # Solo binario, multi-class necesita ajuste
    C = 1.0 / ajustar_alpha(n_features, 'binary_classification')
    print(f'Modelo: LogisticRegression-L1 (clasificación). C usado: {C}')
    model = LogisticRegression(penalty='l1', solver='liblinear', C=C, random_state=RANDOM_STATE, max_iter=10000)

## 4. Estandarización (obligatorio para Lasso)

In [ ]:
# Lasso sensible a escala: importante estandarizar
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

## 5. Ajustar el modelo

In [ ]:
model.fit(X_train_scaled, y_train)
print('Modelo ajustado. Nº de coeficientes ≠ 0:', np.sum(model.coef_!=0))

## 6. Visualización: Importancia de features

In [ ]:
coefs = model.coef_ if target_type=='regression' else model.coef_[0]

coef_df = pd.DataFrame({'feature': X_train.columns, 'coef': coefs}).sort_values('coef')
coef_df['status'] = np.where(coef_df['coef']==0, 'Eliminada', 'Seleccionada')

plt.figure(figsize=(10, int(0.25*len(coef_df))+4))
colors = coef_df['status'].map({'Seleccionada':'green', 'Eliminada':'red'})
plt.barh(coef_df['feature'], coef_df['coef'], color=colors)
plt.title('Coeficientes Lasso: Features Seleccionadas vs Eliminadas')
plt.xlabel('Coeficiente')
plt.axvline(0, color='grey', linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()

## 7. Selección formal de features con SelectFromModel

In [ ]:
sfm = SelectFromModel(model, prefit=True, threshold=1e-6)   # threshold pequeño
selected_mask = sfm.get_support()
selected_features = X_train.columns[selected_mask].tolist()
removed_features = X_train.columns[~selected_mask].tolist()

## 8. Reporte final

In [ ]:
print('--- FEATURE SELECTION REPORT ---')
print(f'Nº de features originales:   {len(feature_names)}')
print(f'Nº de features seleccionadas: {len(selected_features)}')
print(f'Nº de features eliminadas:    {len(removed_features)}')
print('\nFeatures seleccionadas:')
for f in selected_features:
    print(' -', f)

# Tabla resumen
pd.DataFrame({
    'Feature': feature_names,
    'Status': ['Seleccionada' if f in selected_features else 'Eliminada' for f in feature_names],
    'Coeficiente': coefs
}).sort_values(by='Coeficiente', key=np.abs, ascending=False).head(30)

## 9. Guardar lista de features seleccionadas

In [ ]:
selected_path = Path('proyecto_pipeline_machine_learning_muestra/selected_features.csv')
pd.Series(selected_features).to_csv(selected_path, index=False)
print(f'Lista exportada a: {selected_path.resolve()}')

----

# Conclusiones
- La selección automática de features con Lasso depende de la regularización (`alpha`/`C`), que aquí se adaptó dinámicamente según el número de variables.
- Las variables con coeficiente exactamente 0 fueron descartadas porque el modelo juzgó que no aportan valor predictivo dados el resto de las variables y el nivel de regularización.
- La lista exportada puede ser utilizada directamente en el siguiente paso del pipeline de modelado y/o interpretación.